# AI Agent — 판단과 실행을 한 번씩 진행하기

LLM은 다음 행동을 선택하고, 코드는 선택한 도구를 실행합니다.
실행 결과는 현재 상태와 실행 이력에 저장되어 다음 판단에 사용됩니다.

이번 실습에서는 `agent_runner.py`의 `step()`을 직접 호출하며
LLM이 무엇을 선택하고, 실행 결과에 따라 다음 선택이 어떻게 달라지는지 확인합니다.

## step()을 호출하면

1. 목표, 현재 상태, 실행 가능한 행동 목록, 실행 이력을 LLM에 전달합니다.
2. LLM의 판단 결과를 바탕으로 선택한 도구를 실행합니다.
3. 실행 결과를 현재 상태에 반영하고 실행 이력에 기록합니다.
4. 이번 판단과 실행 결과를 노트북에 반환합니다.

LLM이 `action=None`을 반환하면 도구 실행과 상태·이력 갱신 없이
판단 결과와 `result=None`을 반환합니다.

## 실습 방법

초기화 셀은 한 번만 실행하고, `step()` 호출 셀을 반복 실행합니다.
매번 달라지는 판단 결과와 현재 상태, 누적된 실행 이력을 확인하세요.
`action=None`이면 추가 실행을 멈춥니다.

처음부터 다시 실습하려면 초기화 셀을 다시 실행합니다.

1. 초기화

In [1]:
from agent_runner import step

goal = "다음 주 부산 출장 준비"

current_state = {
    "trip_date": None,
    "meeting_time": None,
    "transport_type": None,
    "transport_status": None,
    "transport_message": None,
}

available_actions = [
    "check_calendar",
    "search_transport",
]

history = []

2. 한 단계 실행 - 이 셀을 반복 실행

In [5]:
step_result = step(
  goal=goal,
  current_state=current_state,
  available_actions=available_actions,
  history=history,
)

decision = step_result["decision"]

print("선택한 행동:", decision["action"])
print("실행 인자:", decision["arguments"])
print("선택 이유:", decision["reason"])
print("실행 결과:", step_result["result"])

if decision["action"] is None:
  print("Agent가 종료를 선택했습니다.")

선택한 행동: None
실행 인자: {'trip_date': None, 'meeting_time': None, 'transport_type': None}
선택 이유: BUS 좌석을 이용할 수 있는 것으로 확인되어 교통편 검색을 종료합니다.
실행 결과: None
Agent가 종료를 선택했습니다.


3. 갱신된 상태와 누적 이력 확인

In [6]:
import json

print("[현재 상태]")
print(json.dumps(current_state, ensure_ascii=False, indent=2))

print("\n[실행 이력]")
print(json.dumps(history, ensure_ascii=False, indent=2))

[현재 상태]
{
  "trip_date": "2026-10-08",
  "meeting_time": "14:00",
  "transport_type": "BUS",
  "transport_status": "available",
  "transport_message": "버스 좌석이 있습니다."
}

[실행 이력]
[
  {
    "action": "check_calendar",
    "arguments": {
      "trip_date": null,
      "meeting_time": null,
      "transport_type": null
    },
    "reason": "현재 상태에 출장 날짜와 미팅 시간이 없으므로 캘린더에서 일정을 확인해야 합니다.",
    "result": {
      "trip_date": "2026-10-08",
      "meeting_time": "14:00"
    }
  },
  {
    "action": "search_transport",
    "arguments": {
      "trip_date": "2026-10-08",
      "meeting_time": "14:00",
      "transport_type": "KTX"
    },
    "reason": "출장 날짜와 미팅 시간이 확인되었고 아직 교통편 검색 이력이 없으므로 KTX 좌석을 검색합니다.",
    "result": {
      "transport_type": "KTX",
      "transport_status": "sold_out",
      "transport_message": "KTX 좌석이 매진되었습니다."
    }
  },
  {
    "action": "search_transport",
    "arguments": {
      "trip_date": "2026-10-08",
      "meeting_time": "14:00",
      "transport_type": "BUS"
  